In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import textwrap
import matplotlib.colors as mcolors

sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams['figure.dpi'] = 120

# INTERVENTION_YEAR = 2013
# BASELINE_YEAR = 2019
FILE_NAME = 'data/Final_Imputed_Panel_Data.csv'

print("Setup complete. Ready for data processing.")

Setup complete. Ready for data processing.


In [2]:
df_raw = pd.read_csv(FILE_NAME)

In [3]:
df_raw.head()

,Country Name,Country Code,Year,Access to electricity (% of population),"Air transport, freight (million ton-km)","Air transport, passengers carried",Container port traffic (TEU: 20 foot equivalent units),Control of Corruption: Estimate,Control of Corruption: Number of Sources,Control of Corruption: Percentile Rank,...,Political Stability and Absence of Violence/Terrorism: Estimate,Political Stability and Absence of Violence/Terrorism: Number of Sources,"Railways, goods transported (million ton-km)","Railways, passengers carried (million passenger-km)","Tariff rate, applied, simple mean, all products (%)","Tariff rate, most favored nation, weighted mean, all products (%)","Unemployment, total (% of total labor force) (modeled ILO estimate)","Unemployment, total (% of total labor force) (national estimate)",Region,DID_Group
0,Cambodia,KHM,2001,9.5,4.087,124847.0,NaN,-0.990784,3.0,17.460318,...,-0.722165,3.0,92.0,33.0,16.740,16.490,0.778,2.055,BRI - Southeast Asia,BRI Treatment Group
1,Cambodia,KHM,2002,13.1,4.087,124847.0,NaN,-0.990784,3.0,17.460318,...,-0.722165,3.0,92.0,39.0,16.280,16.280,0.862,1.482,BRI - Southeast Asia,BRI Treatment Group
2,Cambodia,KHM,2003,19.3,3.265,164515.0,NaN,-0.989836,5.0,14.285714,...,-0.652009,3.0,92.0,45.0,16.260,16.470,0.909,0.909,BRI - Southeast Asia,BRI Treatment Group
3,Cambodia,KHM,2004,14.2,3.143,162132.0,NaN,-1.058346,6.0,14.285714,...,-0.407748,4.0,92.0,45.0,15.175,13.775,1.004,7.530,BRI - Southeast Asia,BRI Treatment Group
4,Cambodia,KHM,2005,20.5,1.202,168810.0,253271.0,-1.223740,10.0,10.243902,...,-0.393915,5.0,92.0,45.0,14.090,11.080,1.061,5.439,BRI - Southeast Asia,BRI Treatment Group


In [ ]:
df = df_raw.copy()

df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df = df[(df["Year"] >= 2001) & (df["Year"] <= 2025)].copy()

id_cols = ["Country Name", "Country Code", "Year"]
value_cols = [c for c in df.columns if c not in id_cols]

df = df.sort_values(["Country Name", "Year"])
df = df.drop_duplicates(subset=["Country Name", "Year"], keep="first")

all_years = list(range(2001, 2026))
all_countries = sorted(df["Country Name"].dropna().unique())

full_index = pd.MultiIndex.from_product(
    [all_countries, all_years],
    names=["Country Name", "Year"]
)

country_code_map = (
    df[["Country Name", "Country Code"]]
    .drop_duplicates("Country Name")
    .set_index("Country Name")["Country Code"]
)

df_full = df.set_index(["Country Name", "Year"]).reindex(full_index)
df_full["Country Code"] = df_full.index.get_level_values("Country Name").map(country_code_map)
df_full = df_full.reset_index()

agg_dict = {col: lambda x: list(x) for col in value_cols}
agg_dict["Country Code"] = "first"

df_ts = (
    df_full.sort_values(["Country Name", "Year"])
    .groupby("Country Name", as_index=False)
    .agg(agg_dict)
)

df_ts.head(1)

,Country Name,Access to electricity (% of population),"Air transport, freight (million ton-km)","Air transport, passengers carried",Container port traffic (TEU: 20 foot equivalent units),Control of Corruption: Estimate,Control of Corruption: Number of Sources,Control of Corruption: Percentile Rank,Current account balance (% of GDP),"Current account balance (BoP, current US$)",...,Political Stability and Absence of Violence/Terrorism: Number of Sources,"Railways, goods transported (million ton-km)","Railways, passengers carried (million passenger-km)","Tariff rate, applied, simple mean, all products (%)","Tariff rate, most favored nation, weighted mean, all products (%)","Unemployment, total (% of total labor force) (modeled ILO estimate)","Unemployment, total (% of total labor force) (national estimate)",Region,DID_Group,Country Code
0,Cambodia,"[9.5, 13.1, 19.3, 14.2, 20.5, 27.5, 20.2, 26.4...","[4.087, 4.087, 3.265, 3.143, 1.202, 1.111, 2.0...","[124847.0, 124847.0, 164515.0, 162132.0, 16881...","[nan, nan, nan, nan, 253271.0, 253271.0, 25327...","[-0.990784227848053, -0.990784227848053, -0.98...","[3.0, 3.0, 5.0, 6.0, 10.0, 13.0, 13.0, 14.0, 1...","[17.4603176116943, 17.4603176116943, 14.285714...","[-2.11975415828821, -2.38394601973682, -4.6255...","[-87877926.7863017, -107306836.844281, -233437...",...,"[3.0, 3.0, 3.0, 4.0, 5.0, 6.0, 6.0, 6.0, 6.0, ...","[92.0, 92.0, 92.0, 92.0, 92.0, nan, nan, nan, ...","[33.0, 39.0, 45.0, 45.0, 45.0, 45.0, 45.0, nan...","[16.74, 16.28, 16.26, 15.175, 14.09, 13.73, 12...","[16.49, 16.28, 16.47, 13.775, 11.08, 10.86, 11...","[0.778, 0.862, 0.909, 1.004, 1.061, 1.16, 1.25...","[2.055, 1.4820000000000002, 0.909, 7.53, 5.439...","[BRI - Southeast Asia, BRI - Southeast Asia, B...","[BRI Treatment Group, BRI Treatment Group, BRI...",KHM


In [15]:
rename_map = {
    "Country Name": "country",
    "Country Code": "country_code",
    "Year": "year",

    "Access to electricity (% of population)": "electricity_access",
    "Air transport, freight (million ton-km)": "air_freight",
    "Air transport, passengers carried": "air_passengers",
    "Container port traffic (TEU: 20 foot equivalent units)": "container_traffic",

    "Control of Corruption: Estimate": "corr_control_est",
    "Control of Corruption: Number of Sources": "corr_control_n",
    "Control of Corruption: Percentile Rank": "corr_control_rank",

    "Current account balance (% of GDP)": "cab_gdp_pct",
    "Current account balance (BoP, current US$)": "cab_usd",

    "Export value index (2015 = 100)": "export_value_idx",
    "Export volume index (2015 = 100)": "export_volume_idx",
    "Exports of goods and services (annual % growth)": "exports_growth",
    "Exports of goods and services (current US$)": "exports_usd",

    "GDP growth (annual %)": "gdp_growth",
    "GDP per capita (current US$)": "gdp_pc",
    "GNI per capita, Atlas method (current US$)": "gni_pc_atlas",
    "Gini index": "gini",

    "Government Effectiveness: Estimate": "gov_eff_est",
    "Government Effectiveness: Number of Sources": "gov_eff_n",

    "Import value index (2015 = 100)": "import_value_idx",
    "Import volume index (2015 = 100)": "import_volume_idx",
    "Imports of goods and services (% of GDP)": "imports_gdp_pct",

    "Individuals using the Internet (% of population)": "internet_users_pct",

    "Political Stability and Absence of Violence/Terrorism: Estimate": "pol_stab_est",
    "Political Stability and Absence of Violence/Terrorism: Number of Sources": "pol_stab_n",

    "Railways, goods transported (million ton-km)": "rail_freight",
    "Railways, passengers carried (million passenger-km)": "rail_passengers",

    "Tariff rate, applied, simple mean, all products (%)": "tariff_applied_pct",
    "Tariff rate, most favored nation, weighted mean, all products (%)": "tariff_mfn_pct",

    "Unemployment, total (% of total labor force) (modeled ILO estimate)": "unemp_ilo",
    "Unemployment, total (% of total labor force) (national estimate)": "unemp_national",

    "Region": "region",
    "DID_Group": "did_group"
}

In [16]:
df_ts = df_ts.rename(columns=rename_map)

In [17]:
df_ts.head(1)

,country,electricity_access,air_freight,air_passengers,container_traffic,corr_control_est,corr_control_n,corr_control_rank,cab_gdp_pct,cab_usd,...,pol_stab_n,rail_freight,rail_passengers,tariff_applied_pct,tariff_mfn_pct,unemp_ilo,unemp_national,region,did_group,country_code
0,Cambodia,"[9.5, 13.1, 19.3, 14.2, 20.5, 27.5, 20.2, 26.4...","[4.087, 4.087, 3.265, 3.143, 1.202, 1.111, 2.0...","[124847.0, 124847.0, 164515.0, 162132.0, 16881...","[nan, nan, nan, nan, 253271.0, 253271.0, 25327...","[-0.990784227848053, -0.990784227848053, -0.98...","[3.0, 3.0, 5.0, 6.0, 10.0, 13.0, 13.0, 14.0, 1...","[17.4603176116943, 17.4603176116943, 14.285714...","[-2.11975415828821, -2.38394601973682, -4.6255...","[-87877926.7863017, -107306836.844281, -233437...",...,"[3.0, 3.0, 3.0, 4.0, 5.0, 6.0, 6.0, 6.0, 6.0, ...","[92.0, 92.0, 92.0, 92.0, 92.0, nan, nan, nan, ...","[33.0, 39.0, 45.0, 45.0, 45.0, 45.0, 45.0, nan...","[16.74, 16.28, 16.26, 15.175, 14.09, 13.73, 12...","[16.49, 16.28, 16.47, 13.775, 11.08, 10.86, 11...","[0.778, 0.862, 0.909, 1.004, 1.061, 1.16, 1.25...","[2.055, 1.4820000000000002, 0.909, 7.53, 5.439...","[BRI - Southeast Asia, BRI - Southeast Asia, B...","[BRI Treatment Group, BRI Treatment Group, BRI...",KHM


In [18]:
mapping_df = pd.DataFrame({
    "original_name": list(rename_map.keys()),
    "short_name": list(rename_map.values())
})

mapping_df

,original_name,short_name
0,Country Name,country
1,Country Code,country_code
2,Year,year
3,Access to electricity (% of population),electricity_access
4,"Air transport, freight (million ton-km)",air_freight
5,"Air transport, passengers carried",air_passengers
6,Container port traffic (TEU: 20 foot equivalen...,container_traffic
7,Control of Corruption: Estimate,corr_control_est
8,Control of Corruption: Number of Sources,corr_control_n
9,Control of Corruption: Percentile Rank,corr_control_rank


In [ ]:
df_ts.to_csv('data/WDI_Dataset.csv', index=False)
mapping_df.to_csv('data/Mapping_Dictionary.csv', index=False)
print("cleaned dataset saved as: WDI_Dataset.csv")

cleaned dataset saved as: WDI_Dataset.csv


In [22]:
df_ts.shape, df.shape

((16, 35), (400, 36))